# 00 · Setup compartido

Este notebook **no se ejecuta solo**: cada notebook lo llama con `%run ./00_Setup`.

| Qué deja listo | Para qué |
|---|---|
| Esquema y volumen en Unity Catalog | Un lugar propio para el laboratorio (`lab_tvmaze_m56`) |
| Rutas y nombres de tablas | Un solo sitio donde cambiarlos |
| Funciones del ETL | `download_days()`, `load_bronze()`, `build_silver()`, `validate_silver()`, `build_gold()` |
| Ayudantes de Jobs y pipelines | `create_job()`, `run_job()`, `create_pipeline()`, `run_pipeline()` |
| Ayudantes del SQL warehouse | `find_warehouse()`, `query_warehouse()` |

```
              Unity Catalog gobierna todo: workspace.lab_tvmaze_m56            (módulo 4)
API TVMaze ─► RAW (JSON en el Volumen) ─► BRONZE ─► SILVER ─► GOLD            (módulo 5)
                                          SQL warehouse: los analistas consultan gold (módulo 6)
```

## 1. Ubicación: catálogo, esquema y volumen

In [0]:
import os, time, json, inspect, requests
import pandas as pd
from datetime import date, timedelta
from pathlib import Path
from pyspark.sql import functions as F
from databricks.sdk import WorkspaceClient

CATALOG = spark.sql("SELECT current_catalog()").first()[0]   # 'workspace' en la Free Edition
SCHEMA  = "lab_tvmaze_m56"
VOLUME  = "datos"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.{SCHEMA} COMMENT 'Laboratorio de los módulos 5 y 6: Data Pipelines y Databricks SQL, con Unity Catalog'")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.{SCHEMA}.{VOLUME} COMMENT 'Archivos del laboratorio: los JSON de la API'")
spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA {SCHEMA}")

USER = spark.sql("SELECT current_user()").first()[0]         # tu usuario: recibe las alertas del Job

## 2. Rutas y nombres

In [0]:
# Archivos (Volumen)
BASE     = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
RAW_PATH = f"{BASE}/raw/schedule"        # JSON tal cual llega: schedule_AAAA-MM-DD.json

# Tablas del ETL con notebooks y con el Job (notebooks 04 y 05)
BRONZE = "bronze_schedule"       # crudo, tal como llega
SILVER = "silver_episodes"       # limpio, una fila por episodio
GOLD   = "gold_network_daily"    # episodios por red y día

# Tablas del pipeline declarativo (notebook 06): las mismas tres capas, con el prefijo dlt_
DLT_BRONZE = "dlt_bronze_schedule"
DLT_SILVER = "dlt_silver_episodes"
DLT_GOLD   = "dlt_gold_network_daily"

# Nombres del Job y del pipeline
JOB_NAME      = "lab_tvmaze_m56_workflow"
PIPELINE_NAME = "lab_tvmaze_m56_pipeline"

## 3. Esquema explícito del JSON

Solo los campos que usaremos (regla del módulo 2: no inferir el esquema en cada carga).

In [0]:
ESQUEMA_EPISODIOS = """
    id BIGINT, name STRING, season INT, number INT, airdate STRING, runtime INT,
    rating STRUCT<average: DOUBLE>,
    _embedded STRUCT<
        show: STRUCT<
            id: BIGINT, name: STRING, language: STRING,
            network: STRUCT<name: STRING>,
            webChannel: STRUCT<name: STRING>
        >
    >
"""

## 4. Extracción: de la API a RAW

In [0]:
def download_day(fecha):
    """Descarga un día de la API a RAW. Devuelve 'nuevo', 'existía' o 'falló'."""
    destino = Path(RAW_PATH) / f"schedule_{fecha.isoformat()}.json"
    if destino.exists():
        return "existía"
    try:
        r = requests.get("https://api.tvmaze.com/schedule/web", params={"date": fecha.isoformat()}, timeout=30)
        r.raise_for_status()
    except requests.RequestException:
        return "falló"
    destino.parent.mkdir(parents=True, exist_ok=True)
    destino.write_bytes(r.content)          # el JSON exactamente como lo devolvió la API
    time.sleep(0.5)                         # cortesía con la API
    return "nuevo"


def download_days(inicio, dias):
    """Descarga varios días seguidos y resume el resultado."""
    res = {"nuevo": 0, "existía": 0, "falló": 0}
    for i in range(dias):
        res[download_day(inicio + timedelta(days=i))] += 1
    print(f"Descargados: {res['nuevo']} | Ya existían: {res['existía']} | Fallidos: {res['falló']}")


def raw_files():
    """Nombres de los archivos JSON que hay en RAW, ordenados."""
    if not Path(RAW_PATH).exists():
        return []
    return sorted(f.name for f in dbutils.fs.ls(RAW_PATH))


def write_bad_file():
    """Simula un archivo con errores: un episodio sin id y otro con una fecha que no es fecha."""
    malos = [
        {"id": None,      "name": "Episodio sin id",         "airdate": "2024-03-01",  "runtime": 30},
        {"id": 900000001, "name": "Episodio con fecha mala", "airdate": "no-es-fecha", "runtime": 30},
    ]
    Path(RAW_PATH).mkdir(parents=True, exist_ok=True)
    (Path(RAW_PATH) / "schedule_con_errores.json").write_text(json.dumps(malos), encoding="utf-8")
    print("Archivo schedule_con_errores.json: 2 episodios con errores")

## 5. El ETL en cuatro funciones

Cada función hace **una sola cosa** y reemplaza su tabla completa (`overwrite`): correrla dos veces da el mismo resultado.

In [0]:
def load_bronze():
    """RAW → bronze: lee todos los JSON con el esquema explícito y los guarda tal como llegan."""
    df = (spark.read.schema(ESQUEMA_EPISODIOS).option("multiLine", "true").json(RAW_PATH)
          .select("*",
                  F.col("_metadata.file_name").alias("source_file"),
                  F.current_timestamp().alias("ingested_at")))
    df.write.mode("overwrite").saveAsTable(BRONZE)
    print(f"bronze: {spark.table(BRONZE).count()} filas de {len(raw_files())} archivos")


def build_silver():
    """bronze → silver: aplana el JSON, corrige tipos y descarta las filas sin id o sin fecha."""
    df = (spark.table(BRONZE)
          .select(
              F.col("id").alias("episode_id"),
              F.col("name").alias("episode_name"),
              "season",
              F.col("number").alias("episode_number"),
              F.expr("try_cast(airdate AS DATE)").alias("airdate"),
              F.col("runtime").alias("runtime_min"),
              F.col("rating.average").alias("episode_rating"),
              F.col("_embedded.show.id").alias("show_id"),
              F.col("_embedded.show.name").alias("show_name"),
              F.col("_embedded.show.language").alias("show_language"),
              F.coalesce(F.col("_embedded.show.network.name"),
                         F.col("_embedded.show.webChannel.name"),
                         F.lit("(sin red)")).alias("network_name"),
              "source_file")
          .where("episode_id IS NOT NULL AND airdate IS NOT NULL"))
    df.write.mode("overwrite").saveAsTable(SILVER)
    print(f"silver: {spark.table(SILVER).count()} filas")


def validate_silver(min_rows=1):
    """Revisa las reglas de calidad de silver. Si una no se cumple, lanza un error."""
    df = spark.table(SILVER)
    filas = df.count()
    reglas = {
        f"hay al menos {min_rows} filas": filas >= min_rows,
        "episode_id sin nulos":           df.where("episode_id IS NULL").count() == 0,
        "episode_id sin duplicados":      df.select("episode_id").distinct().count() == filas,
        "airdate sin nulos":              df.where("airdate IS NULL").count() == 0,
    }
    for regla, ok in reglas.items():
        print("✔" if ok else "✘", regla)
    fallidas = [regla for regla, ok in reglas.items() if not ok]
    if fallidas:
        raise Exception(f"Validación fallida: {fallidas}")
    print(f"silver válida: {filas} filas")


def build_gold():
    """silver → gold: episodios por red y día, listo para el negocio."""
    df = (spark.table(SILVER)
          .groupBy("airdate", "network_name")
          .agg(F.count("*").alias("episodes"),
               F.countDistinct("show_id").alias("shows"),
               F.round(F.avg("episode_rating"), 2).alias("avg_rating"),
               F.sum("runtime_min").alias("total_minutes")))
    df.write.mode("overwrite").saveAsTable(GOLD)
    print(f"gold: {spark.table(GOLD).count()} filas")

## 6. Jobs y pipelines como código

Un Job o un pipeline se describe con un **diccionario** (lo mismo que ves en la interfaz con *View as code*).
Estas funciones lo envían a la API de Databricks y esperan el resultado.

In [0]:
w = WorkspaceClient()        # cliente de la API de Databricks; en un notebook ya viene autenticado


def find_job(name):
    """Id del Job con ese nombre (None si no existe)."""
    jobs = w.api_client.do("GET", "/api/2.2/jobs/list", query={"name": name}).get("jobs", [])
    return jobs[0]["job_id"] if jobs else None


def create_job(settings):
    """Crea el Job descrito en `settings`; si ya existe uno con ese nombre, lo actualiza. Devuelve su id."""
    job_id = find_job(settings["name"])
    if job_id:
        w.api_client.do("POST", "/api/2.2/jobs/reset", body={"job_id": job_id, "new_settings": settings})
    else:
        job_id = w.api_client.do("POST", "/api/2.2/jobs/create", body=settings)["job_id"]
    print(f"Job listo: {settings['name']} → {w.config.host}/jobs/{job_id}")
    return job_id


def run_job(job_id, **params):
    """Ejecuta el Job con esos parámetros, espera a que termine y muestra el estado de cada tarea."""
    run_id = w.api_client.do("POST", "/api/2.2/jobs/run-now",
                             body={"job_id": job_id, "job_parameters": params})["run_id"]
    print(f"Corrida en marcha, mírala en: {w.config.host}/jobs/{job_id}/runs/{run_id}")
    while True:
        run = w.api_client.do("GET", "/api/2.2/jobs/runs/get", query={"run_id": run_id})
        if run["state"]["life_cycle_state"] in ("TERMINATED", "SKIPPED", "INTERNAL_ERROR"):
            break
        time.sleep(15)
    estados = {}                                               # si una tarea se reintentó, queda su último intento
    for t in sorted(run["tasks"], key=lambda t: (t.get("start_time") or float("inf"), t.get("attempt_number", 0))):
        estados[t["task_key"]] = t["state"].get("result_state", t["state"]["life_cycle_state"])
    for tarea, estado in estados.items():
        print(f"  {tarea:<15} {estado}")
    print("Resultado del Job:", run["state"].get("result_state", run["state"]["life_cycle_state"]))


def find_pipeline(name):
    """Id del pipeline con ese nombre (None si no existe)."""
    res = w.api_client.do("GET", "/api/2.0/pipelines", query={"filter": f"name LIKE '{name}'"})
    return res["statuses"][0]["pipeline_id"] if res.get("statuses") else None


def create_pipeline(settings):
    """Crea el pipeline descrito en `settings`; si ya existe uno con ese nombre, lo actualiza. Devuelve su id."""
    pipeline_id = find_pipeline(settings["name"])
    if pipeline_id:
        w.api_client.do("PUT", f"/api/2.0/pipelines/{pipeline_id}", body=settings)
    else:
        pipeline_id = w.api_client.do("POST", "/api/2.0/pipelines", body=settings)["pipeline_id"]
    print(f"Pipeline listo: {settings['name']} → {w.config.host}/pipelines/{pipeline_id}")
    return pipeline_id


def run_pipeline(pipeline_id):
    """Lanza una actualización del pipeline y espera a que termine."""
    update_id = w.api_client.do("POST", f"/api/2.0/pipelines/{pipeline_id}/updates", body={})["update_id"]
    print(f"Actualización en marcha, mírala en: {w.config.host}/pipelines/{pipeline_id}/updates/{update_id}")
    while True:
        estado = w.api_client.do("GET", f"/api/2.0/pipelines/{pipeline_id}/updates/{update_id}")["update"]["state"]
        if estado in ("COMPLETED", "FAILED", "CANCELED"):
            break
        time.sleep(15)
    print("Resultado del pipeline:", estado)
    if estado != "COMPLETED":
        raise Exception("El pipeline no terminó bien: abre el enlace de arriba para ver el error")


def show_expectations(pipeline_id):
    """Resultado de las expectativas del pipeline: cuántas filas pasaron y cuántas fallaron cada regla."""
    display(spark.sql(f"""
        SELECT e.dataset AS tabla, e.name AS expectativa,
               sum(e.passed_records) AS pasaron, sum(e.failed_records) AS fallaron
        FROM (
            SELECT explode(from_json(details:flow_progress:data_quality:expectations,
                   'array<struct<name: string, dataset: string, passed_records: bigint, failed_records: bigint>>')) AS e
            FROM event_log('{pipeline_id}')
            WHERE event_type = 'flow_progress'
        )
        GROUP BY ALL
        ORDER BY tabla, expectativa
    """))

## 7. El SQL warehouse

El cómputo de los analistas. Estas funciones lo buscan y le envían consultas por su API, como lo hace una herramienta de BI.

In [0]:
def find_warehouse():
    """El SQL warehouse del workspace (en la Free Edition hay uno solo)."""
    return w.api_client.do("GET", "/api/2.0/sql/warehouses")["warehouses"][0]


def query_warehouse(statement):
    """Envía una consulta al SQL warehouse y devuelve el resultado como DataFrame de pandas."""
    res = w.api_client.do("POST", "/api/2.0/sql/statements", body={
        "warehouse_id": find_warehouse()["id"], "statement": statement,
        "catalog": CATALOG, "schema": SCHEMA, "wait_timeout": "30s"})
    while res["status"]["state"] in ("PENDING", "RUNNING"):        # el warehouse puede estar encendiéndose
        time.sleep(5)
        res = w.api_client.do("GET", f"/api/2.0/sql/statements/{res['statement_id']}")
    if res["status"]["state"] != "SUCCEEDED":
        raise Exception(res["status"].get("error", {}).get("message", res["status"]["state"]))
    columnas = [c["name"] for c in res["manifest"]["schema"]["columns"]]
    return pd.DataFrame(res["result"].get("data_array", []), columns=columnas)

## 8. Ayudantes: tablas del sistema y limpieza

In [0]:
def try_sql(query):
    """Muestra el resultado de una consulta; si la tabla no está disponible en este workspace, avisa sin fallar."""
    try:
        display(spark.sql(query))
    except Exception as e:
        print("No disponible en este workspace:", str(e).splitlines()[0][:150])


def delete_job_and_pipeline():
    """Borra el Job y el pipeline del laboratorio (al borrar el pipeline se borran sus tablas dlt_)."""
    job_id, pipeline_id = find_job(JOB_NAME), find_pipeline(PIPELINE_NAME)
    if job_id:
        w.api_client.do("POST", "/api/2.2/jobs/delete", body={"job_id": job_id})
    if pipeline_id:
        w.api_client.do("DELETE", f"/api/2.0/pipelines/{pipeline_id}")
    print("✔ Job y pipeline borrados")

In [0]:
print("✔ Setup listo")
print(f"  Esquema : {CATALOG}.{SCHEMA}")
print(f"  Archivos: {BASE}")
print(f"  Tablas  : {BRONZE} → {SILVER} → {GOLD}")